# 05 — Ablation Study
Train Logistic Regression on each signal group independently and compare to the full 17-signal model.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from signalrag.config import TRAIN_CSV, TEST_CSV, FEATURE_COLS, LABEL_COL


In [ ]:
train_df  = pd.read_csv(TRAIN_CSV)
test_df   = pd.read_csv(TEST_CSV)
feat_cols = [c for c in FEATURE_COLS if c in train_df.columns and c in test_df.columns]
X_train = train_df[feat_cols].values;  y_train = train_df[LABEL_COL].values
X_test  = test_df[feat_cols].values;   y_test  = test_df[LABEL_COL].values


In [ ]:
groups = {
    "Retrieval\nonly":  [c for c in feat_cols if c in [
        "retriever_score","doc_similarity","ret_score_std",
        "second_passage_sim","n_docs_retrieved","context_length"]],
    "Generation\nonly": [c for c in feat_cols if c in [
        "entropy","avg_token_prob","min_token_prob","var_token_prob",
        "top2_margin","tail_prob","answer_length","answer_is_letter_only"]],
    "Alignment\nonly":  [c for c in feat_cols if c in [
        "context_answer_overlap","embed_sim","rouge_l"]],
    "All 17\ncombined": feat_cols,
}

auroc_scores = {}
for label, cols in groups.items():
    idxs = [feat_cols.index(c) for c in cols]
    sc = StandardScaler()
    Xtr = sc.fit_transform(X_train[:, idxs])
    Xte = sc.transform(X_test[:, idxs])
    lr  = LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced")
    lr.fit(Xtr, y_train)
    auc = roc_auc_score(y_test, lr.predict_proba(Xte)[:, 1])
    auroc_scores[label] = auc
    print(f"  {label.replace(chr(10),' '):<28} AUROC = {auc:.3f}")


In [ ]:
labels = list(auroc_scores.keys())
scores = list(auroc_scores.values())
colors = ["#c3c2b7", "#c3c2b7", "#c3c2b7", "#2a78d6"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, scores, color=colors, width=0.5, zorder=3)
ax.axhline(y=0.824, color="#eb6834", linestyle="--", linewidth=1.5)
ax.text(3.35, 0.829, "TPA (0.824)", color="#eb6834", fontsize=10)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{score:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylim(0.60, 1.00)
ax.set_ylabel("AUROC", fontsize=12)
ax.set_title("Ablation study — signal group contribution (Logistic Regression)", fontsize=12)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines[["top","right"]].set_visible(False)
patches = [
    mpatches.Patch(color="#c3c2b7", label="Individual signal group"),
    mpatches.Patch(color="#2a78d6", label="All 17 combined"),
    mpatches.Patch(color="#eb6834", label="TPA baseline (0.824)", linestyle="--", fill=False),
]
ax.legend(handles=patches, fontsize=10, frameon=False)
plt.tight_layout()
plt.savefig("figure_ablation.png", dpi=300, bbox_inches="tight")
plt.show()
